# Calentamiento S3 · Preguntarle a una tabla
**Estud-IA · Análisis de Datos Básico · Sesión 3**

Hoy practicas las cuatro preguntas que le vas a hacer a tu propia base: **¿cuántas filas hay?, ¿qué es una fila?, ¿qué cuento (COUNT o SUM)? y ¿los grupos suman el total?**

| Parte | Quién | Datos |
|---|---|---|
| **0 · Ejercicio práctico entre todos** | En la sala principal: el profe escribe y ustedes predicen y le dictan la consulta | Ventas de la papelería (8 filas) |
| **A · En tu sala** | Tu equipo: 6 ejercicios con revisión automática ✅ / ❌ | Pedidos de un café (inventados) |
| **B · Ampliación** | Si terminan la A antes de tiempo | Unir los pedidos con el catálogo |
| **Resumen** | Copian el resumen y lo suben por PR a `practica/S3_calentamiento.md` | — |

**Cómo se usa:** corre cada celda con ▶ (o Shift + Enter), de arriba hacia abajo. Donde veas `___`, cámbialo por lo que falta. Si Colab muestra una advertencia, elige **«Ejecutar de todos modos»**: el cuaderno es del curso.

**Roles en la sala:** quien **construye** escribe en el cuaderno y comparte pantalla; quien **verifica** dice qué número espera *antes* de correr la celda. Cambien de rol en cada ejercicio.

In [ ]:
# Celda 1 · Preparar (no cambies nada; solo córrela)
try:
    import duckdb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb
import pandas as pd, json, base64, re

# Papelería (Parte 0): cada fila es una venta de un producto
ventas = pd.DataFrame({
    "id_venta": ["V1","V2","V3","V4","V5","V6","V7","V8"],
    "fecha": ["2026-08-01","2026-08-03","2026-08-05","2026-08-07","2026-09-01","2026-09-03","2026-09-05","2026-09-07"],
    "producto_id": ["P1","P2","P3","P4","P1","P2","P3","P4"],
    "cantidad": [2,5,1,2,3,4,2,1],
    "precio_unitario": [10000,2000,15000,8000,10000,2000,15000,8000],
    "estado": ["completada","completada","completada","cancelada","completada","completada","completada","completada"],
})

# Café La Esquina (Partes A y B, datos inventados)
lineas = pd.DataFrame([
    ("PD01","2026-10-01","C01","CF1",2,2500,"entregado"), ("PD01","2026-10-01","C01","PN1",2,3000,"entregado"),
    ("PD02","2026-10-01","C02","CF2",1,6000,"entregado"),
    ("PD03","2026-10-02","C01","CF1",2,2500,"entregado"), ("PD03","2026-10-02","C01","PN2",1,5000,"entregado"), ("PD03","2026-10-02","C01","JG1",1,7000,"entregado"),
    ("PD04","2026-10-02","C03","CF2",2,6000,"cancelado"), ("PD04","2026-10-02","C03","PN1",3,3000,"cancelado"),
    ("PD05","2026-10-03","C04","CF1",1,2500,"entregado"), ("PD05","2026-10-03","C04","CF2",1,6000,"entregado"), ("PD05","2026-10-03","C04","PN2",2,5000,"entregado"),
    ("PD06","2026-10-03","C02","JG1",2,7000,"entregado"),
    ("PD07","2026-10-04","C05","CF2",3,6000,"entregado"), ("PD07","2026-10-04","C05","PN1",1,3000,"entregado"), ("PD07","2026-10-04","C05","PN2",1,5000,"entregado"), ("PD07","2026-10-04","C05","JG1",1,7000,"entregado"),
], columns=["pedido_id","fecha","cliente_id","producto_id","cantidad","precio_unitario","estado"])

productos = pd.DataFrame({
    "producto_id": ["CF1","CF2","PN1","PN2","JG1"],
    "nombre": ["Tinto","Capuchino","Pandebono","Croissant","Jugo natural"],
    "categoria": ["Café","Café","Panadería","Panadería","Bebidas frías"],
})
# Catálogo con un error: alguien volvió a usar el código PN1 para un producto nuevo
productos_v2 = pd.concat([productos, pd.DataFrame({"producto_id": ["PN1"], "nombre": ["Pandebono x6"], "categoria": ["Panadería"]})], ignore_index=True)

con = duckdb.connect()
for nombre, tabla in {"ventas": ventas, "lineas": lineas, "productos": productos, "productos_v2": productos_v2}.items():
    con.register(nombre, tabla)

def sql(consulta):
    """Corre una consulta SQL y muestra el resultado como tabla."""
    if "___" in consulta:
        print("✋ Todavía hay un ___ en la consulta: cámbialo por lo que falta y vuelve a correr la celda.")
        return None
    try:
        df = con.sql(consulta).df()
        for col in df.columns:   # 21.0 -> 21 para que se lea mejor
            if df[col].dtype.kind == "f" and df[col].dropna().apply(float.is_integer).all():
                df[col] = df[col].astype("Int64")
        return df
    except Exception as e:
        texto = str(e).split("\n")[0]
        print("❌ La consulta tiene un error:", texto)
        if "not found" in texto or "does not exist" in texto:
            print("Pista: revisa el nombre de la tabla o de la columna, y las comillas: 'texto' va entre comillas simples.")
        return None

_R = json.loads(base64.b64decode("eyJBMSI6IDE2LCAiQTIiOiAiYiIsICJBMyI6IDcsICJBNCI6IDE0LCAiQTUiOiAyMSwgIkE2IjogOTk1MDAsICJCMSI6IDAsICJCMiI6IDQyNTAwLCAiQjMiOiAxNn0="))
_PISTAS = {
    "A1": "Cuenta todas las filas, sin filtro: SELECT COUNT(*) FROM lineas.",
    "A2": "Mira el pedido PD03: aparece en tres filas, una por producto.",
    "A3": "16 son las filas (líneas). Un pedido puede tener varias líneas: COUNT(DISTINCT pedido_id).",
    "A4": "Antes del filtro hay 16 filas; las del pedido cancelado (PD04) deben salir.",
    "A5": "COUNT(*) cuenta filas. Las unidades están en la columna cantidad: SUM(cantidad), solo de lo entregado.",
    "A6": "Suma la columna pesos de tu resumen por día. Si no coincide con el total, revisa que los dos tengan el mismo WHERE.",
    "B1": "Si la consulta no devuelve ninguna fila, ningún producto_id se repite: la respuesta es 0.",
    "B2": "Une lineas (solo entregado) con productos y suma pesos de la categoría Café.",
    "B3": "PN1 está dos veces en productos_v2: cada línea de PN1 encuentra dos productos.",
}
_NOMBRES = {"A1": "Filas de la tabla", "A2": "Qué es una fila", "A3": "Pedidos distintos", "A4": "Filas entregadas (después del filtro)",
            "A5": "Unidades entregadas", "A6": "Total en pesos (los grupos suman el total)", "B1": "Claves repetidas en el catálogo",
            "B2": "Pesos de la categoría Café", "B3": "Filas al unir con el catálogo con error"}
_hechos = {}

def revisar(ejercicio, respuesta):
    """Compara tu respuesta con la esperada y te da una pista si no coincide."""
    if respuesta is ___ or respuesta == "___":
        print("✋ Escribe tu respuesta en lugar de ___ y vuelve a correr la celda.")
        return
    esperado = _R[ejercicio]
    if isinstance(esperado, str):
        ok = str(respuesta).strip().lower() == esperado
    else:
        texto = str(respuesta).strip().replace(" ", "")
        if re.fullmatch(r"\d{1,3}(\.\d{3})+", texto):   # 99.500 con punto de miles
            texto = texto.replace(".", "")
        try:
            ok = float(texto.replace(",", ".")) == float(esperado)
        except ValueError:
            ok = False
    _hechos[ejercicio] = (respuesta, ok)
    if ok:
        print(f"✅ {ejercicio} · ¡Correcto! Antes de seguir, digan en voz alta qué significa.")
    else:
        print(f"❌ {ejercicio} · Todavía no. Pista: {_PISTAS[ejercicio]}")

def resumen(equipo, integrantes, aprendimos, duda):
    """Arma el texto para pegar en practica/S3_calentamiento.md"""
    filas = [f"| {k} · {_NOMBRES[k]} | {_hechos[k][0] if k in _hechos else '—'} | {('✅' if _hechos[k][1] else '❌') if k in _hechos else 'sin hacer'} |" for k in _NOMBRES]
    texto = "\n".join([
        f"# Calentamiento S3 · Equipo {equipo}", "",
        f"**Integrantes:** {integrantes}", "",
        "| Ejercicio | Nuestra respuesta | Revisión |", "|---|---|---|", *filas, "",
        f"**Lo que aprendimos:** {aprendimos}", "",
        f"**Una duda que nos quedó:** {duda}",
    ])
    print(texto)

___ = object()   # marcador de los espacios para completar
print("✅ Listo: tablas ventas (8 filas), lineas (16), productos (5) y productos_v2 (6).")

---
## Parte 0 · Ejercicio práctico entre todos (papelería)
Lo construimos juntos en la sala principal. En cada paso: **escribe en el chat el número que esperas**, le dictamos la consulta al profe, la corremos y comprobamos si acertamos. Las cuatro preguntas son las mismas que vas a usar en tu sala y en tu proyecto.

**0.1 · ¿Cuántas filas hay?** Siempre se empieza por aquí: es la primera cifra de control.

In [ ]:
sql("""
SELECT COUNT(*) AS filas
FROM ventas
""")

Resultado esperado: **8**. Ahora mira la tabla completa y completa la frase **«cada fila es…»**: *cada fila es una venta de un producto* (no un cliente, no un día).

In [ ]:
sql("SELECT * FROM ventas")

**0.2 · ¿Qué cuento?** Tres funciones, tres respuestas distintas sobre las mismas filas:

- `COUNT(*)` cuenta **filas** (aquí, ventas).
- `SUM(cantidad)` suma lo que hay en una **columna** (aquí, unidades).
- `COUNT(DISTINCT producto_id)` cuenta **valores distintos** (aquí, productos).

In [ ]:
sql("""
SELECT COUNT(*)                    AS ventas,
       SUM(cantidad)               AS unidades,
       COUNT(DISTINCT producto_id) AS productos
FROM ventas
WHERE estado = 'completada'
""")

Resultado esperado: **7 ventas, 18 unidades, 4 productos**. Las tres son ciertas; cada una responde una pregunta distinta.

**0.3 · Filtra y vuelve a contar.** Antes del filtro había 8 filas; después deben quedar menos. Si quedan más, algo está mal.

In [ ]:
sql("""
SELECT COUNT(*) AS filas_despues_del_filtro
FROM ventas
WHERE estado = 'completada'
""")

Resultado esperado: **8 antes → 7 después** (sale V4, que se canceló).

**0.4 · Resume y comprueba.** `GROUP BY` deja una fila por grupo. La cifra de control: **los grupos suman el total**.

In [ ]:
sql("""
SELECT substr(fecha, 1, 7)            AS mes,
       SUM(cantidad * precio_unitario) AS pesos
FROM ventas
WHERE estado = 'completada'
GROUP BY mes
ORDER BY mes
""")

In [ ]:
sql("""
SELECT SUM(cantidad * precio_unitario) AS pesos_total
FROM ventas
WHERE estado = 'completada'
""")

Resultado esperado: agosto **45.000** + septiembre **76.000** = **121.000**, igual que el total. ✅ Los grupos suman el total.

---
## Parte A · En tu sala: Café La Esquina
Un café (inventado) registra sus pedidos en la tabla `lineas`. Hagan los 6 ejercicios en orden. En cada uno: **predigan, completen los `___`, corran y revisen**.

**A1 · ¿Cuántas filas tiene la tabla `lineas`?**

In [ ]:
sql("""
SELECT COUNT(___) AS filas
FROM lineas
""")

In [ ]:
revisar("A1", ___)   # cambia ___ por el número que te dio

**A2 · ¿Qué es una fila?** Miren la tabla completa (celda de abajo) y elijan:

- **a)** un pedido
- **b)** un producto dentro de un pedido (una línea del pedido)
- **c)** un cliente

In [ ]:
sql("SELECT * FROM lineas ORDER BY pedido_id")

In [ ]:
revisar("A2", "___")   # escribe a, b o c entre las comillas

**A3 · ¿Cuántos pedidos hay?** (Pista: un mismo pedido puede ocupar varias filas.)

In [ ]:
sql("""
SELECT COUNT(DISTINCT ___) AS pedidos
FROM lineas
""")

In [ ]:
revisar("A3", ___)

**A4 · Mirar y filtrar.** Quédense solo con lo **entregado**. ¿Cuántas filas quedan? (Cifra de control: antes del filtro había las filas de A1.)

In [ ]:
sql("""
SELECT pedido_id, producto_id, cantidad
FROM lineas
WHERE estado = '___'
""")

In [ ]:
revisar("A4", ___)   # número de filas que quedaron

**A5 · ¿Cuántas unidades se entregaron?** ¿COUNT o SUM?

In [ ]:
sql("""
SELECT ___(cantidad) AS unidades
FROM lineas
WHERE estado = 'entregado'
""")

In [ ]:
revisar("A5", ___)

**A6 · Resumir y comprobar.** Ventas en pesos **por día** (solo entregado). Después, el total sin agrupar. ¿Los grupos suman el total?

In [ ]:
sql("""
SELECT fecha,
       SUM(cantidad * precio_unitario) AS pesos
FROM lineas
WHERE estado = 'entregado'
GROUP BY ___
ORDER BY fecha
""")

In [ ]:
sql("""
SELECT SUM(cantidad * precio_unitario) AS pesos_total
FROM lineas
WHERE estado = 'entregado'
""")

In [ ]:
revisar("A6", ___)   # el total en pesos, sin puntos (por ejemplo 12345)

---
## Parte B · Ampliación: unir con el catálogo
Solo si terminaron la Parte A. La regla: **antes de unir, cuenta**. Cuenta cada tabla, revisa que la clave no se repita, predice cuántas filas saldrán y, después de unir, vuelve a contar.

**B1 · ¿Se repite algún `producto_id` en `productos`?** Si la consulta no devuelve filas, no se repite ninguno.

In [ ]:
sql("""
SELECT producto_id, COUNT(*) AS veces
FROM productos
GROUP BY producto_id
HAVING COUNT(*) > 1
""")

In [ ]:
revisar("B1", ___)   # ¿cuántos producto_id se repiten?

**B2 · Unir y resumir por categoría.** Cada línea entregada busca su producto en el catálogo. Antes de unir eran 14 filas entregadas: después de unir deben seguir siendo 14, y las categorías deben sumar el total de A6.

In [ ]:
sql("""
SELECT p.categoria,
       COUNT(*)                            AS filas,
       SUM(l.cantidad * l.precio_unitario) AS pesos
FROM lineas AS l
LEFT JOIN productos AS p ON l.producto_id = p.___
WHERE l.estado = 'entregado'
GROUP BY p.categoria
ORDER BY pesos DESC
""")

In [ ]:
revisar("B2", ___)   # pesos de la categoría Café

**B3 · La trampa.** En `productos_v2` alguien volvió a usar el código PN1 para otro producto. Unan las 14 líneas entregadas con ese catálogo: ¿cuántas filas salen?

In [ ]:
sql("""
SELECT COUNT(*) AS filas,
       SUM(l.cantidad * l.precio_unitario) AS pesos
FROM lineas AS l
LEFT JOIN productos_v2 AS p ON l.producto_id = p.producto_id
WHERE l.estado = 'entregado'
""")

In [ ]:
revisar("B3", ___)

Si salieron más de 14 filas, la unión duplicó líneas y los pesos se inflaron **sin ningún error**. Por eso se cuenta antes y después de unir. Se arregla la clave del catálogo, no se esconde el problema.

---
## Resumen para tu PR
Completen los datos y corran la celda. Copien todo el texto que sale y péguenlo en un archivo nuevo de su repositorio: **`practica/S3_calentamiento.md`**, en una rama nueva y con un pull request (paso 4 de la guía).

In [ ]:
resumen(
    equipo      = "E__",
    integrantes = "nombres de pila",
    aprendimos  = "una frase: qué cambia entre COUNT(*), SUM y COUNT(DISTINCT)",
    duda        = "una duda que les quedó (o «ninguna»)",
)